# Production-Grade Ensemble CNN Classifier with Performance Benchmarks

## Visual Exploration, Model Training, Ensembling, and Production Trade-Off Analysis

This notebook demonstrates the end-to-end pipeline for classifying **Cats vs. Dogs** images using multiple diverse Convolutional Neural Network architectures (**CNN 1 Baseline**, **CNN 2 Regularized**, **CNN 3 Deeper**), combining them via **Hard (Majority) Voting**, **Soft Voting**, and **Weighted Soft Voting**, and evaluating whether the ensemble accuracy gains justify the production costs (latency, throughput, memory, parameter size).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Add src to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.data_loader import prepare_dataset, CLASSES
from src.preprocessing import load_full_dataset, preprocess_image, IDX_TO_CLASS
from src.augmentation import get_data_augmentation_layer
from src.models import build_baseline_cnn, build_regularized_cnn, build_deep_cnn
from src.ensemble import EnsembleClassifier
from src.predict import ProductionPredictor

print("Libraries successfully imported!")

## 1. Dataset Overview & Sample Visualization
We use the **Cats vs Dogs** dataset with 100 images (50 cats, 50 dogs), deterministically partitioned into:
- **Training Set**: 70% (35 Cats, 35 Dogs = 70 images)
- **Validation Set**: 15% (7 Cats, 8 Dogs = 15 images)
- **Test Set**: 15% (8 Cats, 7 Dogs = 15 images)

All models evaluate on the exact same test split for fair comparison.

In [ ]:
prepare_dataset(force=False)
(X_train, y_train), (X_val, y_val), (X_test, y_test), test_paths = load_full_dataset()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}   | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

# Display sample test images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < len(X_test):
        ax.imshow(X_test[i])
        label_name = IDX_TO_CLASS[np.argmax(y_test[i])].upper()
        ax.set_title(f"Test #{i+1}: {label_name}", fontweight="bold")
    ax.axis("off")
plt.suptitle("Sample Test Partition Images (128x128 RGB Normalized)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Training-Time Data Augmentation
Data augmentation (Rotation, Zoom, Translation, Contrast) is applied **strictly to the training dataset** to expand the sample footprint and prevent overfitting.

In [ ]:
aug_layer = get_data_augmentation_layer((128, 128))
sample_cat = X_train[0:1]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample_cat[0])
axes[0].set_title("Original Train Image", fontweight="bold")
axes[0].axis("off")

for i in range(1, 5):
    augmented = aug_layer(sample_cat, training=True).numpy()[0]
    axes[i].imshow(np.clip(augmented, 0, 1))
    axes[i].set_title(f"Augmented Variant #{i}")
    axes[i].axis("off")

plt.suptitle("Demonstration of Training-Time Data Augmentation Pipeline", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Comparative Benchmark Results & Trade-Off Analysis
Let's load the generated benchmark results comparing **Individual CNNs** with **Ensemble Strategies** across accuracy, latency, throughput, model size, and memory usage.

In [ ]:
results_path = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "final_comparison.csv"))
if os.path.exists(results_path):
    df_final = pd.read_csv(results_path)
    display(df_final)
else:
    print("Results CSV not found. Please run main.py first.")

## 4. Robustness Stress-Testing Results
We evaluate how individual CNNs and the Ensemble maintain accuracy when images are subjected to:
1. $\pm 30^\circ$ Rotation
2. Gaussian Blur
3. Gaussian Noise
4. Low Illumination (Darkened 0.5x)
5. Over Illumination (Brightened 1.5x)
6. Center Cropping

In [ ]:
robust_path = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "robustness_results.csv"))
if os.path.exists(robust_path):
    df_robust = pd.read_csv(robust_path)
    display(df_robust)
    
    # Plot
    img_path = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "robustness_comparison.png"))
    if os.path.exists(img_path):
        img = Image.open(img_path)
        plt.figure(figsize=(12, 6), dpi=150)
        plt.imshow(img)
        plt.axis("off")
        plt.show()
else:
    print("Robustness CSV not found.")

## 5. Live Production Inference Pipeline & Confidence Gating
We test the `ProductionPredictor` on a sample test image with confidence threshold verification ($> 0.70$ automated approval vs manual review gating).

In [ ]:
predictor = ProductionPredictor(confidence_threshold=0.70)
sample_test_img = X_test[0]
prediction_result = predictor.predict(sample_test_img, ensemble_strategy="soft", include_debug_models=True)

print("=== PRODUCTION INFERENCE RESPONSE ===")
import json
print(json.dumps(prediction_result, indent=2))